# Lumen3D — build a scene on a Colab GPU

This notebook runs the **build** half of Lumen3D on a Colab **GPU**: upload a short
walkthrough video (or a single image), and it reconstructs a language-queryable 3D
scene (a `bundle.pkl` + `scene.ply`). You can then query it right here, or download it
and explore it locally with `lumen3d view`.

Queries are **open-vocabulary** and work on **any object visible anywhere in the clip** —
the pipeline now discovers objects on *every* frame and fuses them in 3D, so something that
only appears late in the video is just as findable as something in the opening shot.

**Before you start:** set the runtime to a GPU — *Runtime → Change runtime type*. A **T4**
is enough for the `DA3-BASE` backbone; for the largest models (`DA3-GIANT-1.1`,
`DA3NESTED-GIANT-LARGE-1.1`) pick an **A100**.


In [ ]:
!nvidia-smi

## 1. Install

Install Lumen3D from GitHub plus the heavy models (Depth Anything 3, SAM 2, and SigLIP
via `transformers`). `addict` is a dependency DA3 forgets to declare, so we add it by hand.

> If the CUDA check in the next cell prints `False`, installing DA3 replaced Colab's GPU
> build of torch with a CPU one. Reinstall the CUDA build to match Colab's CUDA, e.g.
> `!pip install torch --index-url https://download.pytorch.org/whl/cu121`, then
> *Runtime → Restart runtime* and re-run from here.

In [ ]:
!git clone https://github.com/AwaisAdilKhokhar/lumen3d.git
%cd lumen3d
!pip install -q -e .
!pip install -q depth-anything-3 sam2 addict

In [ ]:
import sys, torch
print("Python:", sys.version.split()[0])   # DA3 needs <= 3.13
print("torch :", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## 2. Provide input — a video *or* a single image

Run **one** of the two cells below, depending on what you have:

- **A video** — a short (5–20 s) walkthrough clip; pan slowly around a room or object so
  the scene is seen from several angles. A phone clip is perfect.
- **A single image** — one photo. This gives a *monocular / 2.5D* reconstruction (depth
  from a single view, so no multi-view coverage), but the full query pipeline still runs.

Whichever you run sets a `source` variable that the build cell picks up.

In [ ]:
# Option A — a video (.mp4)
from google.colab import files

uploaded = files.upload()          # pick one .mp4
source = next(iter(uploaded))
print("uploaded:", source)

### …or a single image

Upload one photo instead. No frame extraction — an image is already a single frame.

In [ ]:
# Option B — a single image (.jpg / .png)
from google.colab import files

uploaded = files.upload()          # pick one image
source = next(iter(uploaded))
print("uploaded:", source)

## 3. Build the scene

`reconstruct` runs the full pipeline — DA3 (depth) → SAM 2 (finds objects on **every** frame)
→ 3D instance association (groups each object across frames by shape **and** appearance) →
SigLIP (embeddings) — and writes `<OUT>/bundle.pkl` + `<OUT>/scene.ply`. Set the output folder,
backbone, and stride in the **config cell** below, then run the build.

**Choosing a backbone (fit it to your runtime):**

| Runtime | `MODEL` | Notes |
|---|---|---|
| Free **T4** | `depth-anything/DA3-BASE` | Safe default; keep `STRIDE` high (fewer frames). |
| **A100** | `depth-anything/DA3-GIANT-1.1` | 1B params — largest *plain* giant. |
| **A100** | `depth-anything/DA3NESTED-GIANT-LARGE-1.1` | 2B, absolute largest — *nested* arch, unverified with this pipeline. |

> **Heads-up on quality:** a bigger backbone sharpens depth only so far — the real detail
> ceiling here is the ~504×280 depth resolution forced by the SAM 2 downscale, so GIANT is
> a marginal step over LARGE on the *visible* cloud. Worth it for the demo, but not
> night-and-day.

> **If you hit `CUDA out of memory`:** DA3's multi-view attention grows with the *square* of
> the frame count, so raise `STRIDE` first (fewer frames), then drop to a smaller `MODEL`.
> Reconstruct runs as a subprocess, so an OOM crash frees the GPU — just re-run, no restart.

In [ ]:
# --- build config -----------------------------------------------------------
OUT    = "demo"                          # output scene folder
MODEL  = "depth-anything/DA3-GIANT-1.1"  # 1B "giant" — needs an A100
STRIDE = 30                              # video: keep every Nth frame (ignored for a single image)
# Model ladder (swap MODEL above to move up/down):
#   depth-anything/DA3NESTED-GIANT-LARGE-1.1  # 2B, absolute largest — NESTED arch, unverified here
#   depth-anything/DA3-GIANT-1.1              # 1B, largest plain giant  <-- default
#   depth-anything/DA3-LARGE                  # 0.4B, what we've built with before
#   depth-anything/DA3-BASE                   # 0.1B, fits a free T4
print(f"building {OUT!r} with {MODEL} (stride {STRIDE})")

In [ ]:
!lumen3d reconstruct "{source}" -o {OUT} --stride {STRIDE} --model {MODEL}

## 4. Query the scene

Turn a phrase into a ranked list of matching objects. Scores are low by SigLIP's design —
**rank, don't threshold.** Because objects are discovered on every frame, you can query
anything that appears **anywhere** in the clip, not just the opening shot.


In [ ]:
!lumen3d query {OUT} "a wooden chair"

## 5. Preview the point cloud (optional)

A quick inline 3D scatter of the scene backdrop (subsampled so the browser stays snappy).

> **If this cell raises `TypeError: _reconstruct: First argument must be a sub-type of
> ndarray`:** the kernel started before the pip install, so it's holding an older NumPy than
> the one `reconstruct` used to pickle the scene. Fix: *Runtime → Restart session*, then run
> `%cd lumen3d` (a restart resets the working directory) and re-run this cell. No rebuild
> needed — your scene folder is already on disk. The build and query cells are subprocesses,
> so they're immune; only this in-kernel cell is affected.

In [ ]:
import pickle, numpy as np
import plotly.graph_objects as go

bundle = pickle.load(open(f"{OUT}/bundle.pkl", "rb"))
points, colors = bundle["scene"]

n = min(20000, len(points))                       # subsample for responsiveness
idx = np.random.choice(len(points), n, replace=False)
p, c = points[idx], colors[idx]

fig = go.Figure(go.Scatter3d(
    x=p[:, 0], y=p[:, 1], z=p[:, 2], mode="markers",
    marker=dict(size=1.5, color=[f"rgb({r},{g},{b})" for r, g, b in c]),
))
fig.update_layout(scene=dict(aspectmode="data"), margin=dict(l=0, r=0, t=0, b=0))
fig.show()

## 6. Download the scene

Zip the scene folder and download it. Locally (with Lumen3D installed) explore it in the
browser with:

```bash
lumen3d view demo/
```

In [ ]:
!zip -qr {OUT}.zip {OUT}
from google.colab import files
files.download(f"{OUT}.zip")